In [ ]:
import numpy as np
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv(r"C:\Users\Negus\OneDrive\שולחן העבודה\Excel Files\עותק של microsoft_stocks.csv")

df

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import RidgeCV, Ridge ,LinearRegression,LassoCV

from sklearn.model_selection import train_test_split,cross_val_score , cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_regression
from sklearn.metrics import mean_squared_error , mean_absolute_error

x = df[['High', 'Low', 'Open', 'Volume']] 
y = df['Close']

train_rmse = []
test_rmse = []

for d in range(1, 9):
    polynomial_converter = PolynomialFeatures(degree=d, include_bias=False)

    newX = polynomial_converter.fit_transform(x)


    x_train, x_test, y_train, y_test = train_test_split(newX, y, test_size=0.3, random_state=42)
    
    model = LinearRegression()
    model.fit(x_train, y_train)
    
    train_prediction = model.predict(x_train)
    test_prediction = model.predict(x_test)
    
    train_rmse_value = np.sqrt(mean_squared_error(y_train, train_prediction))
    test_rmse_value = np.sqrt(mean_squared_error(y_test, test_prediction))
    
    train_rmse.append(train_rmse_value)
    test_rmse.append(test_rmse_value)
    
    
print("Train RMSE:", train_rmse)
print("Test RMSE:", test_rmse)

In [ ]:
plt.plot(range(1,9), train_rmse[:9], label='train rmse')
plt.plot(range(1,9), test_rmse[:9], label='test rmse')

plt.ylabel('RMSE')
plt.xlabel('Degree Of Poly')
plt.title('error metrics VS Degree')
plt.legend()
plt.show()
#the best degree is 2

In [ ]:
residual = y_test - test_prediction

sns.scatterplot(x = y_test, y = residual)

plt.axhline(y=0, color='r', ls='--')
plt.show()

In [ ]:
polynomial_converter = PolynomialFeatures(degree=2, include_bias=False)

newX = polynomial_converter.fit_transform(x)
 
scaler = StandardScaler()
scaler.fit(x_train)
scaled_X_train = scaler.transform(x_train)
scaled_X_test = scaler.transform(x_test)

alpha_values = np.linspace(0.1, 3, 15)

ridge_cv_model = RidgeCV(alpha_values, scoring = 'neg_mean_absolute_error', cv=5)
ridge_cv_model.fit(scaled_X_train, y_train)

train_prediction = ridge_cv_model.predict(scaled_X_train)
test_prediction = ridge_cv_model.predict(scaled_X_test)

mae_train = mean_absolute_error(y_train, train_prediction)
mae_test = mean_absolute_error(y_test, test_prediction)

mse_train = mean_squared_error(y_train, train_prediction)
mse_test = mean_squared_error(y_test, test_prediction)

rmse_train = np.sqrt(mse_train)
rmse_test = np.sqrt(mse_test)

print('optimal alpha: ', ridge_cv_model.alpha_)
print('----------------------------------')
print('MAE TRAIN:', mae_train)
print('MAE TEST:', mae_test)
print('----------------------------------')
print('MSE TRAIN:', mse_train)
print('MSE TEST:', mse_test)
print('----------------------------------')
print('RMSE TRAIN:', rmse_train)
print('RMSE TEST:', rmse_test)

In [ ]:
from joblib import dump, load

final_model = RidgeCV()
model.fit(x, y)


dump(ridge_cv_model, 'final_poly_model.joblib')
#ערכים שהם לא פולינומים וממיר לפולינומים
dump(final_poly_converter, 'final_converter.joblib')

dump(scaler, 'scaler.joblib')

loaded_converter = load('final_converter.joblib')
loaded_model = load('final_poly_model.joblib')
loaded_scaler = load('scaler.joblib')

unknown_data = np.array([[250.00, 255.00, 249.50, 23000000], [260.00, 265.00, 259.50, 21500000], [245.00, 250.00, 244.50, 22000000]])

unknown_poly = loaded_converter.transform(unknown_data)

unknown_poly_scaled = loaded_scaler.transform(unknown_poly)

pred = loaded_model.predict(unknown_scaled_poly)

print(pred)
print('---------------------------')
print("Feature coefficients:", loaded_model.coef_)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import RidgeCV, Ridge, LinearRegression, LassoCV, LogisticRegression

from sklearn.model_selection import train_test_split,cross_val_score , cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_regression
from sklearn.metrics import mean_squared_error , mean_absolute_error, accuracy_score, confusion_matrix,  classification_report, recall_score, precision_score

titanic = sns.load_dataset("titanic")
titanic

In [ ]:
new_df = titanic[["age","sex","fare", "survived"]]
new_df

In [ ]:
new_df = new_df.dropna()
new_df

In [ ]:
new_df = pd.get_dummies(new_df, columns = ["sex"], drop_first = False)
new_df

In [ ]:
X = new_df.drop("survived", axis = 1 )
y = new_df["survived"]

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=101)

scaler = StandardScaler()
scaled_x_train = scaler.fit_transform(x_train)
scaled_x_test = scaler.transform(x_test)

log_model = LogisticRegression()
log_model.fit(scaled_x_train, y_train)

y_pred = log_model.predict(scaled_x_test)
y_pred

In [ ]:
log_model.coef_

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1_score = 2 * (precision * recall) / (precision + recall)

print("accuracy score: " , accuracy)
print("recall score: " , recall)
print("precision score: " , precision)
print("f1_score: " , f1_score)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot = True , fmt = 'd' , cmap = plt.cm.viridis )

plt.title("confusion matrix")
plt.xlabel("predicted")
plt.ylabel("True")

plt.show()

In [ ]:
url = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
df = pd.read_csv(url)
df

In [ ]:
#⦁	Print the number of rows and columns in the dataset.
rows, columns = df.shape
rows, columns

In [ ]:
# ⦁	Calculate the mean, median, standard deviation, and other descriptive statistics for each column.
mean_values = df.mean()
median_values = df.median()
std_values = df.std()
describe_values = df.describe()

print("Mean values:\n", mean_values)
print("-----------------------------")
print("\nMedian values:\n", median_values)
print("-----------------------------")
print("\nStandard deviation:\n", std_values)
print("-----------------------------")
print("\nDescriptive statistics:\n", describe_values)

In [ ]:
# ⦁	Plot the distribution of the target variable (`Outcome`).
plt.figure(figsize=(8, 6))
sns.countplot(x='Outcome', data=df, palette='viridis')
plt.title('Distribution of Outcome Variable')
plt.xlabel('Outcome')
plt.ylabel('Count')
plt.show()


In [ ]:
sns.pairplot(df, diag_kind = 'hist')

In [ ]:
correlation_with_outcome = df.corr()['Outcome'].drop('Outcome')

# Plot the correlations
plt.figure(figsize=(10, 6))
correlation_with_outcome.plot(kind='bar', color='skyblue')
plt.title('Correlation between Features and Outcome')
plt.ylabel('Correlation Coefficient')
plt.xlabel('Features')
plt.grid(True)
plt.show()

In [ ]:
# Calculate the correlation between each feature and the outcome label
correlation_with_outcome = df.corr()['Outcome'].drop('Outcome')

# Print the correlation values
print(correlation_with_outcome)


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

image = np.array(
    [
        [[255,255,51],[0,0,0]],
        [[0,0,0],[255,255,51]]
    ],dtype = np.uint8


)
plt.imshow(image)
plt.show()